IMPORTS

In [1]:
import numpy as np

INPUT

In [2]:
states = ["CP","IP"]
symbols = ["cola","ice_t","lem"]

PREPROCESSING

In [3]:
symbol_to_id = {s:i for i,s in enumerate(symbols)}
obs_sequence = ["lem","ice_t","cola"]
obs = [symbol_to_id[s] for s in obs_sequence]

COUNTS

In [4]:
number_of_states = len(states)
number_of_timesteps = len(obs)

INITIAL PROBS

In [5]:
pi = np.array([1.0,0.0])

TRANSITION PROBS

In [6]:
A = np.array([
    [0.7,0.3],
    [0.5,0.5]
])

EMISSION PROBS

In [7]:
B = np.array([
    [0.6,0.1,0.3],
    [0.1,0.7,0.2]
])

INITIALIZATION OF ALPHA

In [8]:
alpha = np.zeros((number_of_timesteps,number_of_states))

In [9]:
for state in range(number_of_states):
    alpha[0][state] = pi[state]*B[state][obs[0]]

COMPUTING ALPHA AT EVERY TIMESTEP

In [10]:
for timestep in range(1,number_of_timesteps):
    for state in range(number_of_states):
        alpha[timestep][state] = sum(
                                    alpha[timestep-1][hidden_state]*A[hidden_state][state] 
                                    for hidden_state in range(number_of_states)
                                    )*B[state][obs[timestep]]

INITIALIZATION OF BETA

In [11]:
beta = np.zeros((number_of_timesteps,number_of_states))

In [12]:
beta[number_of_timesteps-1] = 1

COMPUTING BETA AT EVERY TIMESTEP

In [13]:
for timestep in reversed(range(number_of_timesteps-1)):
    for state in range(number_of_states):
        beta[timestep][state] = sum(
                                    A[state][hidden_state]*B[hidden_state][obs[timestep+1]]*beta[timestep+1][hidden_state] 
                                    for hidden_state in range(number_of_states)
                                    )

INITIALIZATION OF GAMMA

In [14]:
gamma = np.zeros((number_of_timesteps,number_of_states))

COMPUTING GAMMA VALUES

In [15]:
for timestep in range(number_of_timesteps):
    denom = sum(alpha[timestep][hidden_state]*beta[timestep][hidden_state] for hidden_state in range(number_of_states))
    for state in range(number_of_states):
        gamma[timestep][state] = (alpha[timestep][state]*beta[timestep][state]) / denom

COMPUTING STATES

In [16]:
best_states = np.argmax(gamma,axis=1)
decoded_sequence = [states[state] for state in best_states]

INITIALIZATION OF DELTA

In [17]:
delta = np.zeros((number_of_timesteps, number_of_states))
for state in range(number_of_states):
    delta[0][state] = pi[state] * B[state][obs[0]]

COMPUTING DELTA VALUES

In [18]:
for timestep in range(1, number_of_timesteps):
    for current_state in range(number_of_states):
        delta[timestep][current_state] = max(
            delta[timestep - 1][previous_state] * A[previous_state][current_state]
            for previous_state in range(number_of_states)
        ) * B[current_state][obs[timestep]]

DISPLAYING VALUES

In [19]:
print("Alpha:\n", alpha)
print("\nBeta:\n", beta)
print("\nGamma:\n", gamma)

# --------------------
# Final Probability using Forward
forward_prob = np.sum(alpha[number_of_timesteps-1])

# --------------------
# Final Probability using Backward
backward_prob = sum(
    pi[state] * B[state][obs[0]] * beta[0][state]
    for state in range(number_of_states)
)

# --------------------
# Output
print("\nForward Probability:", forward_prob)
print("Backward Probability:", backward_prob)

print("\nBest states:", decoded_sequence)
best_path_prob = np.max(delta[number_of_timesteps - 1])
print("Best path probability:", best_path_prob)

Alpha:
 [[0.3     0.     ]
 [0.021   0.063  ]
 [0.02772 0.00378]]

Beta:
 [[0.105 0.145]
 [0.45  0.35 ]
 [1.    1.   ]]

Gamma:
 [[1.   0.  ]
 [0.3  0.7 ]
 [0.88 0.12]]

Forward Probability: 0.0315
Backward Probability: 0.03149999999999999

Best states: ['CP', 'IP', 'CP']
Best path probability: 0.0189
